In [ ]:
import pandas as pd
from pathlib import Path
import warnings

warnings.simplefilter("ignore")

import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

data_path = Path('/kaggle/input/kaggle-llm-science-exam')

## We'll use `FLAN-T5-base` from Kaggle's Model Hub

You'll probably want to turn on the GPU option for the notebook! (Remember though, since this is a Code competition, you'll need to set Internet to Off for Notebook submissions to the competition.)

In [ ]:
llm_path = '/kaggle/input/flan-t5/pytorch/base/2'

# Use the GPU if one is available, otherwise fall back to CPU
if torch.cuda.is_available():
    device = "cuda:0"
else:
    device = "cpu"

model = T5ForConditionalGeneration.from_pretrained(llm_path).to(device)
tokenizer = T5Tokenizer.from_pretrained(llm_path)

The data is formatted as follows. For each `prompt` (e.g., the question) there are five possible answers labeled `[A-E]`. Only one of the answers is correct.

In [ ]:
test = pd.read_csv(data_path / 'test.csv', index_col='id')
test.head()

## Creating the prompt

How you format your prompt to input to the LLM can make a big difference in the output you get. Here, we try to instruct the LLM to rank all of the options from most likely to least likely.

In [ ]:
preamble = (
    "Answer the following question by outputting the letters A, B, C, D, and E "
    "in order from most likely to be correct to least likely to be correct."
)

In [ ]:
def format_input(df, idx):
    # Pull out the question and the five answer choices for this row
    row = df.loc[idx]
    prompt = row['prompt']
    choice_a = row['A']
    choice_b = row['B']
    choice_c = row['C']
    choice_d = row['D']
    choice_e = row['E']

    # Build the full text we'll feed to the model
    input_text = (
        f"{preamble}\n\n"
        f"{prompt}\n\n"
        f"A) {choice_a}\n"
        f"B) {choice_b}\n"
        f"C) {choice_c}\n"
        f"D) {choice_d}\n"
        f"E) {choice_e}"
    )

    return input_text

This is an example of a formatted question that would be used as input to the LLM.

In [ ]:
print(format_input(test, 0))

In [ ]:
inputs = tokenizer(format_input(test, 0), return_tensors="pt", truncation=True, max_length=512).to(device)
outputs = model.generate(**inputs)
answer = tokenizer.batch_decode(outputs, skip_special_tokens=True)

print(answer)

## Post-processing

You can see from the above that the LLM did not properly follow instructions. You'll need to figure out how to ensure your model provides at least the top three predictions, and have checks and post-processing in place for when they don't (such as in our example!)

This notebook provides a naive and **very** fragile example of how to do this. You'll want to make something more robust!

In [ ]:
def post_process(predictions):
    valid_letters = ['A', 'B', 'C', 'D', 'E']

    # predictions is a list holding one decoded string from the model,
    # e.g. predictions == ['A'] if the model generated a single letter
    decoded_answer = predictions[0]

    # If that string isn't exactly one of our five letters, we can't use it
    if decoded_answer not in valid_letters:
        # Fall back to all five letters and hope for partial credit
        return 'A B C D E'

    # Start the ranking with the one valid letter we found
    ranked_letters = [decoded_answer]

    # Add the remaining letters at the end, so we always submit all 5
    for letter in valid_letters:
        if letter not in ranked_letters:
            ranked_letters.append(letter)

    return ' '.join(ranked_letters)

## Making a submission

We can now make a simple script to make a submission to the competition.

In [ ]:
submission = pd.read_csv(data_path / 'sample_submission.csv', index_col='id')

for idx in test.index:
    # Build the prompt for this question
    input_text = format_input(test, idx)

    # Ask the model to generate an answer
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)
    outputs = model.generate(**inputs)
    answer = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    # Turn the model's raw output into our ranked A-E submission
    submission.loc[idx, 'prediction'] = post_process(answer)

You can include all five possible answers, but only the first three will be counted!

In [ ]:
submission.head()

In [ ]:
submission.to_csv('submission.csv')